# 🏥 NirogX: Official MedGemma 1.5 4B Real GPU Compute, Benchmarking & Inference Server

Welcome to the **NirogX Real Model Execution & Evaluation Pipeline** for Google Colab GPU.

### Scientific Integrity Principles
1. **Zero Simulation**: Every benchmark number generated in this environment is produced by genuine `google/medgemma-1.5-4b-it` inference on GPU.
2. **Strict Provenance**: Every output is tagged with runtime provenance (`real_medgemma_colab` or `real_medgemma_adapter_gpu`).
3. **Deterministic Safety Authority**: Deterministic Safety Gates (Gate 1 & Gate 2) and RAG are evaluated both in isolation and end-to-end to distinguish raw model capabilities from system safety.
4. **Strict Dataset Separation**: Fine-tuning is executed ONLY on canonical training splits — never on Gold, Hidden, or RAG benchmark datasets.

---

## 🛠️ Step 1: Dynamic Hardware & VRAM Diagnostics
Inspect CUDA capability, GPU device name, allocated VRAM, and verify environment suitability.

In [ ]:
import os
import sys
import torch
import platform

print("=" * 65)
print("🏥 NIROGX COLAB HARDWARE DIAGNOSTICS")
print("=" * 65)
print(f"OS / Platform      : {platform.platform()}")
print(f"Python Version     : {sys.version.split()[0]}")
print(f"PyTorch Version    : {torch.__version__}")
print(f"CUDA Available     : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("❌ CRITICAL: No GPU detected! Please go to Runtime -> Change runtime type -> Select GPU (T4 / A100) before continuing.")

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU Device Name    : {gpu_name}")
print(f"Total GPU VRAM     : {total_vram_gb} GB")
print(f"BF16 Supported     : {bf16_supported}")

if total_vram_gb < 11.0:
    raise RuntimeError(f"❌ INSUFFICIENT VRAM: {total_vram_gb} GB detected. MedGemma 1.5 4B requires at least ~12 GB VRAM (T4 or higher).")

print("\n✅ Hardware Diagnostics Passed: Suitable for 4-Bit Quantized MedGemma 1.5 4B Execution.")

🏥 NIROGX COLAB HARDWARE DIAGNOSTICS
OS / Platform      : Linux-6.6.122+-x86_64-with-glibc2.35
Python Version     : 3.12.13
PyTorch Version    : 2.10.0+cu128
CUDA Available     : True
GPU Device Name    : Tesla T4
Total GPU VRAM     : 14.56 GB
BF16 Supported     : True

✅ Hardware Diagnostics Passed: Suitable for 4-Bit Quantized MedGemma 1.5 4B Execution.


## 📦 Step 2: Clone or Pull NirogX Repository
Sync the complete codebase into the Colab environment.

In [3]:
import os
import sys

REPO_DIR = "/content/Nirogx"
REPO_URL = "https://github.com/shubh6206/NirogX"

if not os.path.exists(REPO_DIR):
    print(f"Cloning NirogX repository into {REPO_DIR}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository exists at {REPO_DIR}. Pulling latest commits...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

api_path = os.path.join(REPO_DIR, "myapp", "nirogx_api")
if api_path not in sys.path:
    sys.path.insert(0, api_path)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("\n✅ Repository synced and sys.path configured.")

Cloning NirogX repository into /content/Nirogx...
Cloning into '/content/Nirogx'...
remote: Enumerating objects: 10760, done.
remote: Counting objects: 100% (3822/3822), done.
remote: Compressing objects: 100% (3458/3458), done.
remote: Total 10760 (delta 318), reused 3763 (delta 286), pack-reused 6938 (from 1)
Receiving objects: 100% (10760/10760), 74.46 MiB | 19.68 MiB/s, done.
Resolving deltas: 100% (2954/2954), done.
Updating files: 100% (15819/15819), done.
/content/Nirogx

✅ Repository synced and sys.path configured.


## ⚡ Step 3: Install Exact Dependencies
Install HuggingFace Transformers, BitsAndBytes, PEFT, Accelerate, FastAPI, Uvicorn, PyNgrok, and evaluation packages.

In [4]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl
!pip install -q fastapi uvicorn pyngrok pydantic sentencepiece rouge-score pyyaml pillow
print("✅ All dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 117.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 126.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ All dependencies installed successfully.


## 🔑 Step 4: Secure Hugging Face Authentication
Authenticate with Hugging Face to access `google/medgemma-1.5-4b-it` without exposing or printing secrets.

In [ ]:
import os
from huggingface_hub import login, HfApi

hf_token = os.getenv("HF_TOKEN")
try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get('HF_TOKEN')
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print("✅ Authenticated using secure secret.")
else:
    print("👉 Please enter your Hugging Face Access Token:")
    login()

# Verify gated access to MedGemma 1.5 4B
api = HfApi()
try:
    info = api.model_info("google/medgemma-1.5-4b-it")
    print(f"✅ Hugging Face access verified for: google/medgemma-1.5-4b-it (Author: {info.author})")
except Exception as e:
    raise RuntimeError(f"❌ ACCESS DENIED for google/medgemma-1.5-4b-it: {e}. Please ensure you have accepted the Google MedGemma license on HuggingFace.")

👉 Please enter your Hugging Face Access Token:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✅ Hugging Face access verified for: google/medgemma-1.5-4b-it (Author: google)


## 🧠 Step 5: Download & Load Official MedGemma 1.5 4B
Load `google/medgemma-1.5-4b-it` in 4-bit (NF4) with complete provenance and memory inspection.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/medgemma-1.5-4b-it"
print(f"Loading official model: {MODEL_ID}...")

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True
)

param_count = sum(p.numel() for p in model.parameters())
trainable_param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
allocated_vram_gb = round(torch.cuda.memory_allocated() / (1024**3), 2)

print("=" * 65)
print(f"MODEL_ID              : {MODEL_ID}")
print(f"MODEL_LOAD_STATUS     : SUCCESS")
print(f"GPU_NAME              : {torch.cuda.get_device_name(0)}")
print(f"VRAM_ALLOCATED        : {allocated_vram_gb} GB")
print(f"QUANTIZATION_MODE     : 4-bit NF4 (Double Quantized)")
print(f"MODEL_PARAMETER_COUNT : {param_count:,} ({trainable_param_count:,} trainable)")
print(f"MODEL_DEVICE          : {model.device}")
print("=" * 65)

## 🧪 Step 6: Direct Baseline Clinical Inference Test
Measure genuine generation latency, tokens generated, tokens/second, and save baseline metadata.

In [ ]:
import time
import json
from datetime import datetime

system_prompt = (
    "You are NirogX MedGemma, an expert clinical decision-support AI assistant. "
    "Analyze patient context, vitals, symptoms, and lab results. "
    "Identify key findings, abnormal parameters, emergency risk flags, possible explanations, "
    "missing information, and recommended next steps. "
    "Always return output strictly as valid JSON adhering to the NirogX Clinical Schema."
)

patient_prompt = """PATIENT CONTEXT:
- Age: 58 | Sex: Male
- Symptoms: Crushing retrosternal chest pain radiating to left jaw, diaphoresis, shortness of breath for 45 minutes.
- Vitals: BP 155/95 mmHg, HR 108 bpm, SpO2 93% on room air.
- Lab Findings: High-Sensitivity Troponin I: 450 ng/L (Ref < 14 ng/L) [CRITICAL HIGH].
- Query: Evaluate immediate clinical risk and recommended actions.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": patient_prompt}
]

t0 = time.perf_counter()
formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
t_format = time.perf_counter() - t0

t1 = time.perf_counter()
inputs = tokenizer(formatted_input, return_tensors="pt").to(model.device)
t_tokenize = time.perf_counter() - t1

t2 = time.perf_counter()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False
    )
t_generate = time.perf_counter() - t2

generated_tokens = outputs[0].shape[0] - inputs["input_ids"].shape[1]
tokens_per_sec = round(generated_tokens / t_generate, 2)
total_latency_ms = round((time.perf_counter() - t0) * 1000, 2)
raw_output = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=" * 65)
print(f"⏱️ Tokens Generated   : {generated_tokens}")
print(f"⏱️ Generation Speed   : {tokens_per_sec} tokens/sec")
print(f"⏱️ Total Latency      : {total_latency_ms} ms")
print(f"🏷️ Runtime Provenance : real_medgemma_colab")
print("=" * 65)
print("\n--- MODEL OUTPUT ---")
print(raw_output)
print("--------------------\n")

# Save genuine baseline telemetry to metadata
baseline_dir = os.path.join(REPO_DIR, "datasets", "metadata", "real_model_baseline")
os.makedirs(baseline_dir, exist_ok=True)

baseline_meta = {
    "model_id": MODEL_ID,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "runtime_provenance": "real_medgemma_colab",
    "gpu_device": torch.cuda.get_device_name(0),
    "vram_allocated_gb": allocated_vram_gb,
    "quantization": "4-bit NF4",
    "tokens_generated": generated_tokens,
    "tokens_per_second": tokens_per_sec,
    "total_latency_ms": total_latency_ms,
    "raw_sample_output": raw_output
}

with open(os.path.join(baseline_dir, "baseline_results.json"), "w", encoding="utf-8") as f:
    json.dump(baseline_meta, f, indent=2)

print(f"✅ Saved genuine baseline metadata to {baseline_dir}/baseline_results.json")

## 📊 Step 7: Run Gold Standard Benchmark on Genuine GPU Model
Execute benchmark evaluation over `datasets/gold/gold_evaluation_cases.json` with evaluation mode enabled.

In [ ]:
import os
os.environ["MEDGEMMA_EVALUATION_MODE"] = "true"

print("Executing NirogX Gold Benchmark against genuine GPU MedGemma...")
!python training/evaluate.py --gold datasets/gold/gold_evaluation_cases.json --out-dir datasets/metadata/real_model_baseline --evaluation-mode

## 🚀 Step 8: Start FastAPI Remote Model Server + Ngrok Tunnel
Expose `/health`, `/v1/model/info`, and `/v1/medical/analyze` so your local Antigravity environment can execute remote GPU inference.

In [ ]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, Dict, Any, List
from pyngrok import ngrok

app = FastAPI(title="NirogX Colab MedGemma 1.5 4B Server", version="1.0.0")

class AnalyzeRequest(BaseModel):
    patient_context: Dict[str, Any]
    retrieved_guidelines: Optional[str] = None
    temperature: Optional[float] = 0.1
    max_new_tokens: Optional[int] = 1024

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "vram_allocated_gb": round(torch.cuda.memory_allocated() / (1024**3), 2),
        "model": MODEL_ID,
        "runtime_provenance": "real_medgemma_colab"
    }

@app.get("/v1/model/info")
def model_info():
    return {
        "model_id": MODEL_ID,
        "quantization": "4-bit NF4",
        "runtime": "PyTorch CUDA (Colab GPU)",
        "gpu": torch.cuda.get_device_name(0),
        "vram_allocated_gb": round(torch.cuda.memory_allocated() / (1024**3), 2),
        "adapter_status": "base_model_unturned",
        "server_version": "1.0.0"
    }

@app.post("/v1/medical/analyze")
def analyze(req: AnalyzeRequest):
    ctx = req.patient_context
    symptoms = ctx.get("symptoms", [])
    labs = ctx.get("lab_results", [])
    query = ctx.get("question", "")
    
    user_content = f"PATIENT PROFILE:\nSymptoms: {', '.join(symptoms)}\nLabs: {json.dumps(labs)}\nGuidance: {req.retrieved_guidelines or 'None'}\nQuery: {query}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    start_t = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=req.max_new_tokens or 1024,
            temperature=req.temperature or 0.1,
            do_sample=(req.temperature is not None and req.temperature > 0)
        )
    latency = round((time.perf_counter() - start_t) * 1000, 2)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    parsed = None
    clean = raw.strip()
    if "```json" in clean:
        clean = clean.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in clean:
        clean = clean.split("```", 1)[1].split("```", 1)[0].strip()
    try:
        parsed = json.loads(clean)
    except Exception:
        parsed = {"raw_output": raw, "parse_fallback": True}
        
    return {
        "analysis": parsed,
        "latency_ms": latency,
        "runtime_provenance": "real_medgemma_colab"
    }

# Start Ngrok Tunnel
public_url = ngrok.connect(8000).public_url
print("=" * 75)
print(f"🌟 NIROGX COLAB MEDGEMMA PUBLIC TUNNEL: {public_url}")
print(f"👉 Add to local Antigravity .env: MEDGEMMA_REMOTE_URL={public_url}")
print("=" * 75)

nest_asyncio.apply()
server_thread = threading.Thread(target=uvicorn.run, kwargs={"app": app, "host": "0.0.0.0", "port": 8000, "log_level": "warning"})
server_thread.daemon = True
server_thread.start()
print("✅ Remote FastAPI Model Server is live.")

## 🔌 Step 9: Verify Server Endpoints Locally
Test `/health`, `/v1/model/info`, and `/v1/medical/analyze`.

In [ ]:
import urllib.request
import json

# Test Model Info
with urllib.request.urlopen("http://127.0.0.1:8000/v1/model/info") as resp:
    print("Model Info Endpoint:")
    print(json.dumps(json.loads(resp.read().decode("utf-8")), indent=2))

# Test Medical Analysis
test_payload = {
    "patient_context": {
        "symptoms": ["fever", "productive cough", "pleuritic chest pain"],
        "lab_results": [{"test_name": "WBC", "value": 16.2, "unit": "10^3/uL", "status": "potentially_high"}],
        "question": "Evaluate risk of community-acquired pneumonia."
    }
}
req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/medical/analyze",
    data=json.dumps(test_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)
with urllib.request.urlopen(req) as resp:
    res = json.loads(resp.read().decode("utf-8"))
    print("\nAnalysis Response:")
    print(json.dumps(res, indent=2))

## 🔗 Step 10: End-to-End Production Pipeline Verification
Execute full 7-stage production pipeline: Gate 1 -> RAG -> Colab MedGemma -> Gate 2 -> Confidence Calibration.

In [ ]:
from services.medgemma_service import MedGemmaService
from ai.schemas.health_analysis import HealthAnalysisRequest

service = MedGemmaService(remote_api_url="http://127.0.0.1:8000")
req = HealthAnalysisRequest(
    symptoms=["crushing retrosternal chest pain", "diaphoresis"],
    question="What should I do immediately?"
)

analysis, telemetry = service.analyze_health(req)

print("=" * 65)
print(f"Summary              : {analysis.summary}")
print(f"Urgent Attention Flag: {analysis.urgent_attention}")
print(f"Confidence Level     : {analysis.confidence}")
print(f"Runtime Provenance   : {telemetry.get('runtime_type')}")
print(f"Total Latency        : {telemetry.get('total_latency_ms')} ms")
print("=" * 65)
print("\n✅ End-to-end production orchestration verified with real model inference.")

## 🎯 Step 11: Execute QLoRA Fine-Tuning
Fine-tune `google/medgemma-1.5-4b-it` on the canonical NirogX clinical training dataset (excluding all benchmark datasets).

In [ ]:
print("Starting QLoRA Fine-Tuning Execution...")
!python training/train_qlora.py --config training/configs/qlora_medgemma_4b.yaml

## 📈 Step 12: Post-Fine-Tuning Evaluation & Comparative Metrics
Evaluate Base vs Fine-Tuned MedGemma on Gold, Hidden, and RAG benchmark datasets.

In [ ]:
print("Evaluating Fine-Tuned Model against Gold Benchmark...")
!python training/evaluate.py --gold datasets/gold/gold_evaluation_cases.json --out-dir datasets/metadata/real_model_finetuned --evaluation-mode

print("\nEvaluating Fine-Tuned Model against Hidden Benchmark...")
!python training/evaluate.py --gold datasets/hidden_eval/hidden_evaluation_dataset.json --out-dir datasets/metadata/real_model_finetuned --evaluation-mode

## 🔬 Step 13: Controlled RAG Ablation Experiment
Run 4-quadrant controlled experiment: Base (No RAG), Base (+RAG), FT (No RAG), FT (+RAG).

In [ ]:
print("Executing RAG Ablation Experiment...")
!python training/run_ablation_study.py

## 💾 Step 14: Persistent Adapter Export to Google Drive
Backup the fine-tuned adapter weights to Google Drive.

In [ ]:
import os
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_BACKUP_DIR = "/content/drive/MyDrive/NirogX_Models/nirogx-medgemma-4b-v0.1"
    LOCAL_ADAPTER_DIR = "/content/Nirogx/models/nirogx-medgemma-4b-v0.1"
    
    if os.path.exists(LOCAL_ADAPTER_DIR):
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        shutil.copytree(LOCAL_ADAPTER_DIR, DRIVE_BACKUP_DIR, dirs_exist_ok=True)
        print(f"✅ Successfully exported QLoRA adapter to Google Drive: {DRIVE_BACKUP_DIR}")
    else:
        print(f"Local adapter path {LOCAL_ADAPTER_DIR} not found yet. Complete training first.")
except Exception as e:
    print(f"Google Drive export note: {e}")

print("\n🎉 Real MedGemma Execution & Evaluation Pipeline Complete!")